In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, current_date

In [0]:
catalog_name = spark.conf.get("pipeline.catalog_name", "dbr_dev")
schema_name = spark.conf.get("pipeline.schema_name", "weather_bronze")
batch_volume_path = f"/Volumes/{catalog_name}/{schema_name}/raw/weather-batch/"

In [0]:
@dlt.table(
    name="weather_bronze.ly_rainfall_data",
    comment="Bronze table for batch rainfall data with Auto Loader.",
    table_properties={"quality": "bronze"}
)
def ly_rainfall_data_bronze():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") #schema evolution
        .load(batch_volume_path)
        .selectExpr("*", "_metadata.file_name as source_filename")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
    )